# Minimal Example for Calculating the Elastic Response of a Simply Supported Beam

This notebook illustrates one possible way of calculating the response of a simply supported beam, using as many assumptions and simplifications as necessary to keep it simple enoguh so that a wide range of people would be able to follow it.

## Euler-Bernoulli beam theory

The response of a linearly elastic Bernoulli (Euler-Bernoulli) beam is governed by the following differential equation:

$$
EI \frac{d^4 w(x)}{dx^4} = q(x)
$$

where:

- $E$ is the Young's modulus of the material
- $I$ is the second moment of area of the cross section
- $w(x)$ is the transverse displacement of the beam at position $x$
- $q(x)$ is the distributed load per unit length

Put simply, the product $EI$ measures the resistance of a beam against bending. The higher the $EI$, the harder it is to bend the beam.

The boundary value problem (BVP) is completed by specifying conditions on the boundary. For a simply supported beam of length $L$, the boundary conditions are:

$$
w(0) = 0 \qquad \text{(zero displacement at left end)}
$$
$$
w(L) = 0 \qquad \text{(zero displacement at right end)}
$$
$$
M(0) = 0 \qquad \text{(zero bending moment at left end)}
$$
$$
M(L) = 0 \qquad \text{(zero bending moment at right end)}
$$

where $M(x) = -EI \frac{d^2 w(x)}{dx^2}$ is the bending moment.

## Navier's solution

The solution adopted in this project is due to Navier. The basic idea is to write all functions involved in the DE as a sum of sinusoidal functions.

The general solution for the displacement $w(x)$ under a distributed load $q(x)$ is:

$$
w(x) = \sum_{n=1}^{\infty} \frac{q_n L^4}{EI n^4 \pi^4} \sin\left(\frac{n \pi x}{L}\right)
$$

where $q_n$ are the Fourier coefficients of the load distribution $q(x)$:

$$
q_n = \frac{2}{L} \int_0^L q(x) \sin\left(\frac{n \pi x}{L}\right) dx
$$

This integral can be calculated exactly for some kinds of loads. In the general case, numerical integration is required.

The Navier solution leverages the orthogonality of sine functions to represent both the load and the displacement as Fourier series. This approach is particularly powerful for simply supported beams because the sine functions automatically satisfy the boundary conditions ($w(0) = w(L) = 0$ and $M(0) = M(L) = 0$). By projecting the load $q(x)$ onto these basis functions, we obtain the coefficients $q_n$, which directly determine the amplitude of each mode in the displacement response. The higher the mode number $n$, the smaller its contribution due to the $n^4$ term in the denominator, reflecting the beam's resistance to higher-frequency bending.

## Define a cross-section and calculate its properties

In [5]:
from sectionproperties.analysis import Section
from sectionproperties.pre.library import i_section
from sectionproperties.pre import Material
from pydantic import BaseModel, Field

In [6]:
# define material properties for structural steel
material = Material(
    name="Structural Steel",
    elastic_modulus=210e3,   # in MPa
    poissons_ratio=0.3,      # dimensionless
    yield_strength=250,      # in MPa
    density=7.85e-6,          # in kg/mm^3
    color="gray"
)

# define I-section geometry
geometry = i_section(
    d=300,                  # depth in mm
    b=150,                  # flange width in mm
    t_f=15,                 # flange thickness in mm
    t_w=10,                 # web thickness in mm
    r=12,                   # root radius in mm
    n_r=8,                  # number of points to define the root radius
)

# assign material to geometry
geometry.material=material

# generate finite element mesh
geometry.create_mesh(mesh_sizes=[4])  # mesh size in mm

# create section object for analysis
section = Section(geometry=geometry)

In [7]:
class SectionStiffnessProperties(BaseModel):
    """
    Data class representing key stiffness properties of a cross section.
    """
    A: float = Field(..., description="Area of the section with SI units [m^2].", gt=0)
    ksx: float = Field(..., description="Shear correction factor in x direction (dimensionless).", gt=0)
    ksy: float = Field(..., description="Shear correction factor in y direction (dimensionless).", gt=0)
    Ixx: float = Field(..., description="Second moment of area about x-axis with SI units [m^4].", gt=0)
    Iyy: float = Field(..., description="Second moment of area about y-axis with SI units [m^4].", gt=0)
    Ixy: float = Field(..., description="Product moment of area with SI units [m^4].", gt=0)


def calculate_section_properties(section: Section) -> dict:
    """Extract key geometric properties of the section."""

    # calculate geometric and warping properties
    section.calculate_geometric_properties()
    section.calculate_warping_properties()

    # retrieve properties
    area = section.get_area()
    shear_area_x, shear_area_y = section.get_eas(e_ref=material)
    Ixx, Iyy, Ixy = section.get_eic(e_ref=material)

    ksx = shear_area_x / area
    ksy = shear_area_y / area

    return SectionStiffnessProperties(
        A=area,
        ksx=ksx,
        ksy=ksy,
        Ixx=Ixx,
        Iyy=Iyy,
        Ixy=Ixy,
    )
    
    
props: SectionStiffnessProperties = calculate_section_properties(section)

## Calculate the response of a simply supported beam

SectionStiffnessProperties(A=7327.397797144078, ksx=0.5447893243348102, ksy=0.39155218714420476, Ixx=110094531.791528, Iyy=8468301.22926223, Ixy=4.929315476190476e-07)